# Big Daddy — Single Mixed Workload Analysis

One heavy mixed workload run across all policy variants. Same style as `bench_analysis.ipynb`:
- **Chart 0**: Absolute hit rate by policy.
- **Chart 1**: Hit rate Δ (% vs median) — deviation from the workload median so relative differences stand out.

CSV: `../big_daddy_results.csv` from `scripts/bench_big_daddy.sh`. Images save to `analysis/graphs/`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from pathlib import Path

# Paths: CSV at repo root, images under analysis/graphs
CSV_PATH = Path('../big_daddy_results.csv')
IMG_DIR = Path('graphs')
IMG_DIR.mkdir(exist_ok=True)

# Redirect savefig to graphs/
_original_savefig = plt.savefig
def _savefig_into_graphs(*args, **kwargs):
    if args:
        fname = args[0]
        p = Path(fname)
        if not p.is_absolute() and p.suffix:
            args = (str(IMG_DIR / p.name), *args[1:])
    return _original_savefig(*args, **kwargs)
plt.savefig = _savefig_into_graphs

df = pd.read_csv(CSV_PATH)
print(f'Loaded {len(df)} rows from {CSV_PATH}')
print(df.columns.tolist())

In [ ]:
# Single workload: add a dummy preset so we reuse the same grouped_bar as bench_analysis
df['preset'] = 'big_daddy'

# Derived columns
df['hit_pct'] = df['hit_rate'] * 100

# Median hit rate over this single workload (all policies, all runs)
median_hit = df['hit_pct'].median()
df['hit_rel_pct'] = (df['hit_pct'] / median_hit - 1.0) * 100.0

# Canonical policy order and colors (same as bench_analysis; subset by presence)
POLICIES = [
    'basic_lru', 'arc', 'lfu',
    'lru_2q', 'lru_2q_small', 'lru_2q_large',
    'lecar', 'lecar_fast', 'lecar_slow',
    'cacheus', 'cacheus_lru_biased', 'cacheus_lfu_biased',
    'decision_tree', 'decision_tree_deep', 'decision_tree_fast',
]
present = df['policy'].unique().tolist()
POLICIES = [p for p in POLICIES if p in present]
if len(present) != len(POLICIES):
    for p in present:
        if p not in POLICIES:
            POLICIES.append(p)

PRESETS = ['big_daddy']
PRESET_LABELS = {'big_daddy': 'Big Daddy'}

POLICY_COLORS = {
    'basic_lru': '#4C72B0', 'arc': '#55A868', 'lfu': '#C44E52',
    'lru_2q': '#8172B2', 'lru_2q_small': '#A89CCC', 'lru_2q_large': '#5C4B9B',
    'lecar': '#E07B39', 'lecar_fast': '#F5A623', 'lecar_slow': '#A0522D',
    'cacheus': '#17BECF', 'cacheus_lru_biased': '#2ECC71', 'cacheus_lfu_biased': '#0E7F8A',
    'decision_tree': '#E74C3C', 'decision_tree_deep': '#922B21', 'decision_tree_fast': '#FF8C94',
}
df = df[df['policy'].isin(POLICIES)]
print(f'Policies: {len(POLICIES)}')

In [ ]:
def grouped_bar(ax, metric, ylabel, title=None, fmt='{:.1f}', scale=1.0, annotate=False):
    """Draw grouped bar chart: one band (big_daddy), one bar per policy. Same contract as bench_analysis."""
    n_presets = len(PRESETS)
    n_policies = len(POLICIES)
    group_width = 0.85
    bar_w = group_width / n_policies
    agg = df.groupby(['preset', 'policy'])[metric].mean() * scale
    xs = np.arange(n_presets, dtype=float)
    for i, policy in enumerate(POLICIES):
        offset = (i - n_policies / 2 + 0.5) * bar_w
        vals = [agg.get((p, policy), 0.0) for p in PRESETS]
        color = POLICY_COLORS.get(policy, '#888888')
        bars = ax.bar(xs + offset, vals, width=bar_w * 0.95, color=color, alpha=0.88,
                     edgecolor='white', linewidth=0.4, label=policy)
        if annotate:
            for bar, v in zip(bars, vals):
                if v != 0:
                    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(), fmt.format(v),
                            ha='center', va='bottom', fontsize=6, rotation=90)
    ax.set_xticks(xs)
    ax.set_xticklabels([PRESET_LABELS[p] for p in PRESETS], fontsize=10)
    ax.set_ylabel(ylabel, fontsize=11)
    if title:
        ax.set_title(title, fontweight='bold', fontsize=12)
    ax.grid(axis='y', alpha=0.3, linestyle='--')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.set_xlim(-0.5, n_presets - 0.5)

def policy_legend(fig, ncol=5):
    handles = [mpatches.Patch(color=POLICY_COLORS.get(p, '#888888'), label=p) for p in POLICIES]
    fig.legend(handles=handles, loc='lower center', ncol=ncol, fontsize=8.5,
               frameon=True, framealpha=0.9, bbox_to_anchor=(0.5, -0.01))

print('Helpers defined.')

## Chart 0 — Hit Rate (%)

Absolute hit rate by policy (mean over runs).

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
grouped_bar(ax, 'hit_pct', 'Hit Rate (%)', title='Big Daddy — Hit Rate by Policy')
policy_legend(fig)
fig.tight_layout(rect=[0, 0.12, 1, 1])
plt.savefig('big_daddy_hit_rate.png', dpi=150, bbox_inches='tight')
plt.show()

## Chart 1 — Hit Rate Δ (% vs median)

Deviation from the **median** hit rate across all policies (same idea as bench_analysis).
Y-axis: percentage above/below median (e.g. +20 = 20% above median).

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
grouped_bar(ax, 'hit_rel_pct', 'Hit Rate Δ (% vs median)',
            title='Big Daddy — Hit Rate Δ by Policy (normalized)',
            fmt='{:+.0f}%', scale=1.0, annotate=True)
ax.axhline(0.0, color='black', linewidth=0.8, linestyle='--')
policy_legend(fig)
fig.tight_layout(rect=[0, 0.12, 1, 1])
plt.savefig('big_daddy_hit_rate_delta.png', dpi=150, bbox_inches='tight')
plt.show()